In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r'D:\project_mfjm\candidate_job_pairs.csv')

In [3]:
df.head()

,job_id,resume_id,label,r_role,r_seniority,r_years_experience,r_industry,r_education,r_skills,r_summary,r_experience_bullets,j_job_title,j_seniority,j_industry,j_must_have_skills,j_nice_to_have_skills,j_description,j_responsibilities,j_requirements
0,J_001992,R_004958,1,Operations Manager,Senior,7,FinTech,High School,Asana|KPIs|Project Planning|Risk Management|St...,Operations Manager with 7 years of experience ...,Delivered results using structured workflows a...,Operations Manager,Mid,E-commerce,Asana|Jira|Process Improvement|Risk Management...,Stakeholder Communication,We are hiring a Operations Manager to support ...,Deliver high-quality work aligned with goals a...,Relevant experience and ability to learn quick...
1,J_000712,R_002421,1,Software Engineer,Mid,5,Gaming,High School,JavaScript|Unit Testing|Git|CI/CD|Docker|REST ...,Software Engineer with 5 years of experience i...,Delivered results using structured workflows a...,Full Stack Engineer,Mid,Retail,Databases|Docker|JavaScript|Python|Git,Java,We are hiring a Full Stack Engineer to support...,Deliver high-quality work aligned with goals a...,Relevant experience and ability to learn quick...
2,J_001744,R_000122,0,Technical Product Manager,Mid,2,SaaS,BSc,Scrum|Analytics|Agile|Roadmap|PRD,Technical Product Manager with 2 years of expe...,Delivered results using structured workflows a...,Customer Support Specialist,Junior,FinTech,Ticketing|Documentation|Root Cause Analysis|Co...,Troubleshooting,We are hiring a Customer Support Specialist to...,Deliver high-quality work aligned with goals a...,Relevant experience and ability to learn quick...
3,J_002183,R_003850,0,FP&A Analyst,Junior,1,Travel,MSc,Cash Flow|Budgeting|Variance Analysis|Valuatio...,FP&A Analyst with 1 years of experience in Tra...,Delivered results using structured workflows a...,Customer Support Specialist,Senior,Travel,Documentation|Zendesk|SLA|Root Cause Analysis,Customer Satisfaction,We are hiring a Customer Support Specialist to...,Deliver high-quality work aligned with goals a...,Relevant experience and ability to learn quick...
4,J_002309,R_003074,0,Customer Success Associate,Junior,2,Travel,BA,Documentation|SLA|Escalations|Zendesk|Root Cau...,Customer Success Associate with 2 years of exp...,Delivered results using structured workflows a...,Business Development Manager,Mid,Travel,Outbound Outreach|Negotiation|Discovery Calls|...,Closing,We are hiring a Business Development Manager t...,Deliver high-quality work aligned with goals a...,Relevant experience and ability to learn quick...


In [4]:
df.columns

Index(['job_id', 'resume_id', 'label', 'r_role', 'r_seniority',
       'r_years_experience', 'r_industry', 'r_education', 'r_skills',
       'r_summary', 'r_experience_bullets', 'j_job_title', 'j_seniority',
       'j_industry', 'j_must_have_skills', 'j_nice_to_have_skills',
       'j_description', 'j_responsibilities', 'j_requirements'],
      dtype='object')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 19 columns):
 #   Column                 Non-Null Count   Dtype 
---  ------                 --------------   ----- 
 0   job_id                 150000 non-null  object
 1   resume_id              150000 non-null  object
 2   label                  150000 non-null  int64 
 3   r_role                 150000 non-null  object
 4   r_seniority            150000 non-null  object
 5   r_years_experience     150000 non-null  int64 
 6   r_industry             150000 non-null  object
 7   r_education            150000 non-null  object
 8   r_skills               150000 non-null  object
 9   r_summary              150000 non-null  object
 10  r_experience_bullets   150000 non-null  object
 11  j_job_title            150000 non-null  object
 12  j_seniority            150000 non-null  object
 13  j_industry             150000 non-null  object
 14  j_must_have_skills     150000 non-null  object
 15  

In [9]:
import requests

url = "https://www.arbeitnow.com/api/job-board-api"

response = requests.get(url)
data = response.json()

jobs = data["data"]

In [20]:
import re
import html
from bs4 import BeautifulSoup


def clean_text(text):
    if not text:
        return ""

    # Convert HTML entities
    text = html.unescape(text)

    # Remove escaped backslashes before HTML tags
    text = text.replace("\\<", "<").replace("\\>", ">")

    # Remove HTML tags
    text = BeautifulSoup(text, "html.parser").get_text(" ")

    # Remove markdown-style links
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)

    # Remove remaining URLs
    text = re.sub(r'https?://\S+', '', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


def clean_job(job):
    return {
        "slug": job.get("slug", ""),
        "company_name": clean_text(job.get("company_name", "")),
        "job_title": clean_text(job.get("title", "")),
        "description": clean_text(job.get("description", "")),
        "remote": bool(job.get("remote", False)),
        "url": job.get("url", ""),
        "skills": [
            clean_text(tag)
            for tag in job.get("tags", [])
        ],
        "job_types": [
            clean_text(job_type)
            for job_type in job.get("job_types", [])
        ],
        "location": clean_text(job.get("location", "")),
        "created_at": job.get("created_at")
    }

In [21]:
job = clean_job(jobs[25])

In [22]:
job

{'slug': 'remote-senior-technical-project-manager-pmo-all-genders-289446',
 'company_name': 'Contabo',
 'job_title': 'Senior Technical Project Manager - PMO (all genders)',
 'description': "Your creative field We are looking for a full-time, permanent Technical Project Manager (all genders) to join our PMO team as soon as possible. We embrace a remote-first approach (within Germany), but you can choose whether you’d like to work in a hybrid model or entirely on-site, depending on your proximity to one of our locations (Berlin, Hamburg, Cologne, Munich). As a Senior Technical Project Manager, you are a cornerstone of strategy execution at Contabo – and you're joining at a time when a lot is moving strategically. Alongside the continued evolution of our existing cloud infrastructure, we are driving initiatives that will expand our portfolio well beyond what exists today over the coming years. This is exactly where your role comes in: embedded in our PMO, you take end-to-end ownership of 